In [2]:
!pip install PyPDF2 python-docx scikit-learn nltk
import PyPDF2
import docx
import os
import nltk
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
nltk.download('punkt')
nltk.download('punkt_tab')   # 🔥 important fix
nltk.download('stopwords')
from nltk.corpus import stopwords
from google.colab import files
uploaded = files.upload()
def extract_text_from_pdf(file_path):
    text = ""
    with open(file_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text()
    return text
def extract_text_from_docx(file_path):
    doc = docx.Document(file_path)
    return "\n".join([para.text for para in doc.paragraphs])
def extract_text(file_path):
    if file_path.endswith('.pdf'):
        return extract_text_from_pdf(file_path)
    elif file_path.endswith('.docx'):
        return extract_text_from_docx(file_path)
    elif file_path.endswith('.txt'):
        with open(file_path, 'r') as f:
            return f.read()
    else:
        return ""
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()   # ✅ safer than nltk.word_tokenize
    words = [w for w in words if w not in stop_words]
    return " ".join(words)
SKILLS_DB = [
    "python", "java", "c++", "machine learning", "deep learning",
    "data science", "tensorflow", "pytorch", "nlp",
    "sql", "excel", "communication", "leadership",
    "html", "css", "javascript", "react", "node"
]
def extract_skills(text):
    found_skills = []
    for skill in SKILLS_DB:
        if skill in text:
            found_skills.append(skill)
    return found_skills
print("\nEnter Job Description:\n")
job_description = input()
job_description_clean = clean_text(job_description)
resume_data = []
for file_name in uploaded.keys():
    text = extract_text(file_name)
    cleaned_text = clean_text(text)
    skills = extract_skills(cleaned_text)
    resume_data.append({
        "name": file_name,
        "text": cleaned_text,
        "skills": skills
    })
documents = [job_description_clean] + [r['text'] for r in resume_data]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)
similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
print("\n📊 Resume Ranking:\n")
results = []
for i, resume in enumerate(resume_data):
    results.append({
        "name": resume['name'],
        "score": similarity_scores[i],
        "skills": resume['skills']
    })
results = sorted(results, key=lambda x: x['score'], reverse=True)
for r in results:
    print("📄 Resume:", r['name'])
    print("🔹 Match Score:", round(r['score']*100, 2), "%")
    print("🔹 Skills Found:", ", ".join(r['skills']) if r['skills'] else "None")
    print("-"*50)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving FAANGPath_Simple_Template__2_ (1).pdf to FAANGPath_Simple_Template__2_ (1) (1).pdf

Enter Job Description:

Ai engineer

📊 Resume Ranking:

📄 Resume: FAANGPath_Simple_Template__2_ (1) (1).pdf
🔹 Match Score: 2.75 %
🔹 Skills Found: python, java, machine learning, tensorflow, pytorch, nlp, sql, communication
--------------------------------------------------
